#CafChem tools for running a basic chat loop with a small HuggingFace model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MauricioCafiero/CafChem_Teaching_Notebooks/blob/main/notebooks/HF_agent_step0_CafChem.ipynb)

## This notebook allows you to:
- Load a small (sub-1GB) instruct model from HuggingFace (SmolLM2-360M-Instruct).
- Run a basic chat loop: user message in, model reply out, conversation remembered.
- See how the chat template turns the conversation into model input.

## Requirements:
- This notebook will install transformers.
- Runs quickly on any Colab runtime (GPU or CPU); the model is only ~720MB.

### install libraries

In [ ]:
!pip install -q transformers accelerate

### import libraries

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using", device)

### set up model

In [ ]:
# SmolLM2-360M-Instruct: Hugging Face's own small chat model, ~360M
# parameters, ~720MB download (bf16 weights).
MODEL_REPO = "HuggingFaceTB/SmolLM2-360M-Instruct"
MAX_NEW_TOKENS = 512

SYSTEM_PROMPT = "You are a helpful assistant running on Google Colab."

tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO)
model = AutoModelForCausalLM.from_pretrained(MODEL_REPO, torch_dtype="auto").to(device)

## Define the chat function

The conversation is a list of messages, each with a role (system, user, or assistant). Each turn:
1. Add the user's message to the conversation.
2. Turn the conversation into model input and generate a reply.
3. Decode only the new tokens (everything after the prompt) back to text.
4. Remember the reply so the model has context for the next turn.

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

def respond(user_input):
  """Add a user message to the conversation and print the model's reply."""
  messages.append({"role": "user", "content": user_input})

  inputs = tokenizer.apply_chat_template(
      messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
  output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                              pad_token_id=tokenizer.pad_token_id)
  content = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1] :],
                             skip_special_tokens=True).strip()

  messages.append({"role": "assistant", "content": content})
  print(f"bot: {content}\n")

## Chat with the model

Call `respond()` with a message. Each call remembers what was said before, so the model has context:

In [ ]:
respond("Hi! What is your name?")

In [ ]:
respond("What did I just ask you?")

## Interactive chat

The same loop as a chatbot: type a message at the prompt, type `quit` to stop. Colab will show an input box above the cell's output.

In [ ]:
print("Ready. Type a message (or 'quit').\n")

while True:
  user_input = input("you: ").strip()

  if user_input.lower() in ("quit", "exit"):
    break
  if not user_input:
    continue

  respond(user_input)

## Look at the raw conversation

This is exactly what gets sent to the model each turn (via the chat template):

In [ ]:
messages